In [1]:
import json 
import requests

In [2]:
from bs4 import BeautifulSoup 
import urllib3

In [3]:
MSAMB_URL = "https://www.msamb.com/ApmcDetail/APMCPriceInformation#CommodityDistrictGird"

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time


In [5]:
import selenium
selenium.__version__

'4.40.0'

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service

# Specify the path to your ChromeDriver executable
 # Example path for macOS/Linux

brave_path = '/opt/brave.com/brave/brave'

# Set up Chrome options and specify the binary location
options = webdriver.ChromeOptions()
options.binary_location = brave_path
options = webdriver.ChromeOptions()
options.binary_location = brave_path
options.add_argument('--headless=new')           # Do not open a visible browser window
options.add_argument('--disable-gpu')            # Disable GPU hardware acceleration
options.add_argument('--no-sandbox')             # Bypass OS security model (required for headless on Linux)
options.add_argument('--disable-dev-shm-usage')  # Overcome limited resource problems
options.add_argument('--blink-settings=imagesEnabled=false') # Do not load images (Huge speed boost)
options.page_load_strategy = 'eager'
# Initialize the WebDriver with the specified options and service
service = Service()
driver = webdriver.Chrome(options = options)
# Now you can run your tests as usual
driver.get(MSAMB_URL)
print(driver.title)



Titles


In [7]:
wait = WebDriverWait(driver,10)

In [8]:
lang_element = wait.until(EC.element_to_be_clickable((By.ID,"language")))
lang_select = Select(lang_element)

In [9]:
if "English" not in lang_select.first_selected_option.text:
        print("🌐 Switching to English...")
        lang_select.select_by_visible_text("English")
        time.sleep(3) # Wait for reload
        wait.until(EC.presence_of_element_located((By.ID, "drpCommodities")))

🌐 Switching to English...


In [10]:
print("selecting commodity")

dropdown = Select(driver.find_element(By.ID,"drpCommodities"))

selecting commodity


In [11]:
found=False
crop_name ="wheat"
for opt in dropdown.options:
    if crop_name.upper() in opt.text.upper():
        dropdown.select_by_visible_text(opt.text)
        found=True
        break

    

In [12]:
table_element = driver.find_element(By.CSS_SELECTOR, "table.table.custom-table")
table_html = table_element.get_attribute('outerHTML')

In [13]:
import pandas as pd 
from io import StringIO
df = pd.read_html(StringIO(table_html))[0]

In [14]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(df[:100]) 


,APMC,Variety,Unit,Quantity,Lrate,Hrate,Modal
0,18/02/2026,18/02/2026,18/02/2026,18/02/2026,18/02/2026,18/02/2026,18/02/2026
1,KOLHAPUR-LAXMIPURI,----,QUINTAL,194,2500,5000,3750
2,RAHURI-VAMBORI,----,QUINTAL,9,2300,2651,2563
3,PACHORA,----,QUINTAL,80,2190,2740,2561
4,KARANJA,----,QUINTAL,475,2465,2465,2465
5,KARMALA,----,QUINTAL,7,2200,2500,2500
6,TULJAPUR,----,QUINTAL,70,2425,2500,2450
7,RAHATA,----,QUINTAL,12,2450,2580,2515
8,VADVANI,----,QUINTAL,19,2655,3011,2800
9,LASALGAON-NIPHAD,2189,QUINTAL,64,2251,2640,2451


In [16]:
def _clean_data(df):
    """
    Cleans messy APMC data using Vectorized Forward Fill (Zero Loops).
    """
    date_series = pd.to_datetime(self.df.iloc[:, 0], format="%d/%m/%Y", errors='coerce')
    
    # Create Date Column & Forward Fill
    self.df['Date'] = date_series
    self.df['Date'] = self.df['Date'].ffill()
    
    # Drop the rows that were actually just headers (where date_series was NOT NaT)
    # AND drop rows where APMC is null
    clean = self.df[date_series.isna() & self.df.iloc[:, 0].notna()].copy()
    
    # Rename columns standardly
    clean.rename(columns={
        0: 'APMC', 1: 'Variety', 2: 'Unit', 3: 'Quantity',
        4: 'Min_Price', 5: 'Max_Price', 6: 'Modal_Price'
    }, inplace=True)
    
    # Clean Numeric Data (Remove commas)
    cols = ['Quantity', 'Min_Price', 'Max_Price', 'Modal_Price']
    for col in cols:
        clean[col] = pd.to_numeric(clean[col].astype(str).str.replace(",", ""), errors='coerce')
        
    return clean

In [17]:
_clean_data(df)

,Date,APMC,Variety,Unit,Quantity,Min_Price,Max_Price,Modal_Price
1,2026-02-17,KOLHAPUR-LAXMIPURI,----,QUINTAL,218,2600,4800,3700
2,2026-02-17,RAHURI-VAMBORI,----,QUINTAL,4,2370,2370,2370
3,2026-02-17,PACHORA,----,QUINTAL,85,2200,2500,2451
4,2026-02-17,KARMALA,----,QUINTAL,8,2200,2400,2300
5,2026-02-17,PALGHAR(BEVUR),----,QUINTAL,65,3425,3425,3425
...,...,...,...,...,...,...,...,...
422,2026-02-11,NAGPUR,SHARBATI,QUINTAL,144,3200,3500,3425
423,2026-02-11,NANDED,SHARBATI,QUINTAL,8,2400,2400,2400
424,2026-02-11,HINGOLI,SHARBATI,QUINTAL,26,2500,3060,2780
425,2026-02-11,KALYAN,SHARBATI,QUINTAL,3,2800,3600,3200


In [34]:


# 1. Run in background (Fixes "DevToolsActivePort" & GUI errors)
# chrome_options.add_argument("--headless=new") 
# # 2. Required for most server/notebook environments
# chrome_options.add_argument("--no-sandbox")
# # 3. Fixes memory issues on some systems
# chrome_options.add_argument("--disable-dev-shm-usage")

# --- INITIALIZE DRIVER ---
# This automatically downloads the correct driver for your installed Chrome version
brave_path = '/opt/brave.com/brave/brave'

# Set up Chrome options and specify the binary locatio

try:
    url =MSAMB_URL
    print("Opening url")
    driver.get(url)

    wait = WebDriverWait(driver, 15)

    lang_element = wait.until(EC.element_to_be_clickable((By.ID,"language")))
    lang_select = Select(lang_element)

    if "English" not in lang_select.first_selected_option.text:
        print("🌐 Switching to English...")
        lang_select.select_by_visible_text("English")
        time.sleep(3) # Wait for reload
        wait.until(EC.presence_of_element_located((By.ID, "drpCommodities")))

    # 3. Select Commodity
    print(f"🔍 Selecting '{crop_name}'...")
    dropdown = Select(driver.find_element(By.ID, "drpCommodities"))
    found = False
    for opt in dropdown.options:
        if crop_name.upper() in opt.text.upper():
            dropdown.select_by_visible_text(opt.text)
            found = True
            break

    time.sleep(3) # Let the table populate
        
    # Locate all rows in the specific commodity table body
    rows = driver.find_elements(By.XPATH, "//tbody[@id='tblCommodity']/tr")
    print(f"📥 Found {len(rows)} market entries. Extracting...")
    
    full_data = []
    
    for row in rows:
        cols = row.find_elements(By.TAG_NAME, "td")
        # Ensure it's a valid data row (sometimes headers repeat or empty rows exist)
        if len(cols) >= 7:
            entry = {
                "APMC": cols[0].text.strip(),
                "Variety": cols[1].text.strip(),
                "Unit": cols[2].text.strip(),
                "Quantity": cols[3].text.strip(),
                "Min_Price": cols[4].text.strip(),
                "Max_Price": cols[5].text.strip(),
                "Modal_Price": cols[6].text.strip(),
                "Date": "Today" # You can scrape the date header if needed
            }
            full_data.append(entry)

except Exception as e:
    print(e)

WebDriverException: Message: unknown error: cannot find Chrome binary
Stacktrace:
#0 0x5bc24fe674e3 <unknown>
#1 0x5bc24fb96c76 <unknown>
#2 0x5bc24fbbd757 <unknown>
#3 0x5bc24fbbc029 <unknown>
#4 0x5bc24fbfaccc <unknown>
#5 0x5bc24fbfa47f <unknown>
#6 0x5bc24fbf1de3 <unknown>
#7 0x5bc24fbc72dd <unknown>
#8 0x5bc24fbc834e <unknown>
#9 0x5bc24fe273e4 <unknown>
#10 0x5bc24fe2b3d7 <unknown>
#11 0x5bc24fe35b20 <unknown>
#12 0x5bc24fe2c023 <unknown>
#13 0x5bc24fdfa1aa <unknown>
#14 0x5bc24fe506b8 <unknown>
#15 0x5bc24fe50847 <unknown>
#16 0x5bc24fe60243 <unknown>
#17 0x7c4eae89caa4 <unknown>
#18 0x7c4eae929c6c <unknown>
